# Week 4: Noise Sensitivity Analysis and Threshold Tuning

**Day 1 - Build a genuine held-out test set + clean baseline**

Goal for today: create a test set the model has truly never seen, train an evaluation model on the remaining 80% using the same approach and hyperparameters that won in Week 3, and record clean (no-noise) performance as today's reference line.



In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import lightgbm as lgb
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    recall_score, precision_score, f1_score,
    roc_auc_score, average_precision_score
)

import joblib

np.random.seed(42)

In [ ]:
DATA_PATH = "ai4i_fused_week2.csv"  # your Week 2 output

df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

## Step 1 - Recreate the Week 3 feature set

Same leakage-aware feature list as Week 3 - the five failure-mode flags directly define `Machine failure`, so they stay excluded.

In [ ]:
target = "Machine failure"
leak_cols = ["TWF", "HDF", "PWF", "OSF", "RNF"]
drop_cols = ["UDI", "Product ID", "Type", "timestamp", target] + leak_cols

feature_cols = [c for c in df.columns if c not in drop_cols]
X = df[feature_cols]
y = df[target]

print(f"Using {len(feature_cols)} features")

## Step 2 - The winning approach and best hyperparameters from Week 3

In [ ]:
# Set this to match whatever won in YOUR Week 3 three-way comparison
WINNING_APPROACH = "class_weight"  # or "smote"

tuning_df = pd.read_csv("week3_hyperparameter_tuning_results.csv")
best_row = tuning_df.loc[tuning_df["f1"].idxmax()]
best_params = {
    "num_leaves": int(best_row["num_leaves"]),
    "learning_rate": float(best_row["learning_rate"]),
    "n_estimators": int(best_row["n_estimators"]),
}
print("Winning approach:", WINNING_APPROACH)
print("Best hyperparameters:", best_params)

In [5]:
def build_model(params, approach, y_train=None):
    if approach == "smote":
        return ImbPipeline([
            ("smote", SMOTE(random_state=42)),
            ("clf", lgb.LGBMClassifier(random_state=42, verbose=-1, **params)),
        ])
    else:
        neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
        spw = neg / pos
        return lgb.LGBMClassifier(random_state=42, verbose=-1, scale_pos_weight=spw, **params)

## Step 3 - A fresh, genuinely held-out 80/20 split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"Train: {len(X_train)} rows ({y_train.mean()*100:.3f}% failures)")
print(f"Test:  {len(X_test)} rows ({y_test.mean()*100:.3f}% failures)")

## Step 4 - Fit the evaluation model, get the clean baseline

This exact fitted model is what every noise level in Day 2 will be tested against - nothing about the model changes this week, only the input data.

In [ ]:
X_train_cleaned = X_train.rename(columns=lambda x: x.replace('[', '_').replace(']', '_').replace('<', '_').replace(' ', '_'))
X_test_cleaned = X_test.rename(columns=lambda x: x.replace('[', '_').replace(']', '_').replace('<', '_').replace(' ', '_'))

eval_model = build_model(best_params, WINNING_APPROACH, y_train)
eval_model.fit(X_train_cleaned, y_train)

proba_clean = eval_model.predict_proba(X_test_cleaned)[:, 1]
pred_clean = eval_model.predict(X_test_cleaned)

clean_baseline = {
    "noise_level": 0.0,
    "recall": recall_score(y_test, pred_clean),
    "precision": precision_score(y_test, pred_clean, zero_division=0),
    "f1": f1_score(y_test, pred_clean),
    "roc_auc": roc_auc_score(y_test, proba_clean),
    "pr_auc": average_precision_score(y_test, proba_clean),
}

print("Clean (no-noise) baseline on held-out test set:")
for k, v in clean_baseline.items():
     print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

## Step 5 - Save everything Day 2 needs

In [ ]:
joblib.dump(eval_model, "week4_eval_model.joblib")
X_test.to_csv("week4_X_test.csv", index=False)
y_test.to_csv("week4_y_test.csv", index=False)
pd.DataFrame([clean_baseline]).to_csv("week4_clean_baseline.csv", index=False)

print("Saved week4_eval_model.joblib, week4_X_test.csv, week4_y_test.csv, week4_clean_baseline.csv")